# Train the Jarvie wake/sleep models on RunPod

Uses `wakeword_pipeline.py` (same folder), which caches every stage on the network volume:

| stage | reruns only when you change | A40 time (rough) |
|---|---|---|
| **gen**: speech synthesis | phrases, negatives, sample counts, speeds | ~1-2 h per model, once |
| **aug**: augmentation + features | the augmentation variant (A, B, C, ...) | ~1 h per model per variant |
| **train**: train, export, evaluate | training settings (a "trial") | depends on `steps` |

So A and B share one speech-synthesis run, C later reuses it too, and tuning trials only retrain.
Re-running a cell with unchanged settings, in this session or a later one, just loads the cached result.

### Pod setup (once)
1. **Storage > Network Volume**: create one, **100 GB** (ACAV data ~16 GB, speech clips, features, models),
   in a region that has A40s.
2. **Deploy a pod**: GPU **A40** (48 GB), template **RunPod PyTorch** (PyTorch >= 2.5, Python >= 3.11),
   attach the network volume (mounted at `/workspace`). Open **JupyterLab**.
3. Create `/workspace/wakeword/` and upload **this notebook and `wakeword_pipeline.py`** into it (drag & drop).
   Everything under `/workspace` survives stopping the pod; everything else (installed packages) doesn't, so
   cell 1 runs every session.

### Each session
Run cells 1-4, then whichever run you want (5: A, 6: B, 7: C, 8: tuning). **Stop the pod when done**; the
volume (and all cached stages) stays. Terminating a pod is also fine, as long as you keep the network volume.

## 1. Install (every session)
The pod's own disk is reset when it stops, so packages are reinstalled each time; the pip cache on the volume makes that quick.

In [ ]:
import os

os.environ["PIP_CACHE_DIR"] = "/workspace/.pip-cache"  # on the volume, so reinstalling is fast

# System programs (not pip packages):
#   espeak-ng: Piper turns text into phonemes by calling the espeak-ng command (speech synthesis)
#   ffmpeg:    decodes m4a/mp3 room recordings for variant C
!apt-get -qq update && apt-get -qq install -y espeak-ng ffmpeg > /dev/null

# Python packages:
#   [train]: torch, librosa, scipy, audiomentations, webrtcvad, nltk, pronouncing, cmudict, ... (imported mid-run)
#   [eval]:  matplotlib, imported only by the evaluation step
#   onnx, onnxscript: needed by the export step, but not in livekit's extras
!pip install -q "livekit-wakeword[train,eval]" "setuptools<81" onnx onnxscript pandas

!espeak-ng --version

## 2. Preflight check
Fails in seconds if anything is missing. Continue when it prints `missing: none`.
Run this **first after every kernel restart**: it sets the thread limits before torch loads.

In [ ]:
import importlib
import shutil
import sys
from pathlib import Path

# Size numpy / torch / OpenMP thread pools to the pod's real CPU quota before anything loads
# them: the container reports the host's 128 cores, and 128 threads on ~12 real CPUs made
# feature extraction crawl (~1 clip/s). Importing the pipeline sets this up. If you change it,
# restart the kernel; libraries only read it when first imported.
sys.path.insert(0, str(Path.cwd()))
import wakeword_pipeline  # noqa: E402

print(
    f"CPUs available to this pod: {wakeword_pipeline.available_cpus()} "
    f"(os.cpu_count() says {os.cpu_count()})"
)

# Must be set before torch first uses the GPU: lets PyTorch grow memory in pieces instead of needing one
# big free block (avoids "CUDA out of memory ... Skipping 50 clips" during speech synthesis).
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import nltk  # noqa: E402
import torch  # noqa: E402

modules = [
    "torch",
    "torchaudio",
    "soundfile",
    "librosa",
    "scipy.signal",
    "audiomentations",
    "torch_audiomentations",
    "webrtcvad",
    "nltk",
    "pronouncing",
    "cmudict",
    "sklearn",
    "tqdm",
    "yaml",
    "rich",
    "huggingface_hub",
    "matplotlib",
    "onnxruntime",
    "onnxruntime.quantization",
    "onnx",
    "onnxscript",
    "pandas",
    "livekit.wakeword",
]
missing = []
for m in modules:
    try:
        importlib.import_module(m)
    except Exception as e:
        missing.append(f"{m}: {e}")
for program in ["espeak-ng", "ffmpeg"]:
    if shutil.which(program) is None:
        missing.append(f"{program}: program not found (apt-get install {program})")
nltk.download("cmudict", quiet=True)  # livekit would otherwise download this mid-run

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name}, {props.total_memory / 1e9:.0f} GB")
else:
    print("GPU: NONE - deploy the pod with a GPU")
free_gb = shutil.disk_usage("/workspace").free / 1e9
print(
    f"/workspace free: {free_gb:.0f} GB"
    + ("  <- low, consider a bigger volume" if free_gb < 40 else "")
)
print("missing:", missing or "none")

## 3. Settings (edit here)
Everything that defines a run lives in this cell. Changing something only reruns the stages it affects:
- `PHRASES`, `NEGATIVES`, `GENERATION`: speech synthesis and everything after it
- `pipe.augmentations`: that variant's augmentation and training
- `TRAINING`, or keyword overrides in `pipe.run(...)`: training only

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))  # wakeword_pipeline.py sits next to this notebook
from wakeword_pipeline import Pipeline  # noqa: E402

ROOT = Path("/workspace/wakeword")

PHRASES = {
    # Every phrase stays under ~2 s even when spoken slowly: livekit trains on 2.0 s windows
    # and crops the *start* of longer clips.
    "jarvie_wake": [
        "jarvie, wake up",
        "jarvie, help me",
        "jarvie, wakie wakie",
        "jarvie, are you up",
        "jarvee, wake up",
    ],
    "jarvie_sleep": ["jarvie, go to sleep", "jarvie, sleep now", "jarvee, go to sleep"],
}

# Phrases that must NOT trigger. livekit's automatic hard negatives only swap sounds inside single
# words, so the phrases without "jarvie" have to be listed here.
NEGATIVES = {
    "jarvie_wake": [
        "wake up",
        "wake up wake up",
        "wakie wakie",
        "help me",
        "can you help me",
        "i need some help",
        "jarvie",
        "jacky wake up",
        "harvey wake up",
        "jarvis wake up",
        "barbie help me",
        "carvey wakie wakie",
        "you up",
        "are you up",
        "jacky are you up",
        "jarvis are you up",
    ],
    "jarvie_sleep": [
        "sleep",
        "go to sleep",
        "time to sleep",
        "i need some sleep",
        "jarvie",
        "jarvie keep",
        "jarvie sweep",
        "jacky go to sleep",
        "harvey go to sleep",
    ],
}
# Each model must also ignore the other model's phrases.
NEGATIVES["jarvie_wake"] += PHRASES["jarvie_sleep"]
NEGATIVES["jarvie_sleep"] += PHRASES["jarvie_wake"]

GENERATION = dict(
    n_samples=20000,
    n_samples_val=4000,
    n_background_samples=2000,
    n_background_samples_val=500,
    length_scales=[0.65, 0.75, 0.85, 1.0, 1.1, 1.2],  # speaking rate; smaller = faster
    tts_batch_size=32,  # 50 ran out of memory once on an A40; use 25 on a 16 GB GPU (speed only)
)

TRAINING = dict(
    steps=80000,
    model={"model_size": "medium"},  # "small" for the Raspberry Pi build
    max_negative_weight=3000,
    target_fp_per_hour=0.1,
)

pipe = Pipeline(
    root=ROOT, phrases=PHRASES, negatives=NEGATIVES, generation=GENERATION, training=TRAINING
)

# Augmentation variants (edit or add your own, e.g. pipe.augmentations["D"] = {...}).
for name, spec in pipe.augmentations.items():
    print(name, spec)

### Optional: the stage-2 "jarvie" model
A third model that fires on the name alone, used while the bot runs, so it can tell "Jarvie" from "Jarvis" by
sound. Run this cell **after cell 3 and before running any variant**: it adds the model to `PHRASES`/`NEGATIVES`,
and every later `pipe.run(...)` then also generates its synthetic speech (once) and trains it. Its real recordings
come from the `name-*` and `neg-name-*` sections of `recording_prompts.txt`.

In [ ]:
PHRASES["jarvie_name"] = ["jarvie", "hey jarvie", "okay jarvie", "jarvee"]
NEGATIVES["jarvie_name"] = [
    "jarvis",
    "hey jarvis",
    "okay jarvis",
    "harvey",
    "marvie",
    "carvey",
    "garvey",
    "barbie",
    "charlie",
    "jar of",
    "starving",
    "carving",
    "marvel",
    "java",
    "jarring",
]
# Leave the wake/sleep lists alone: changing them would regenerate those models' speech (hours) and
# invalidate B's cache. They already have "jarvie" as a negative, and your recorded name-* clips
# become negatives for them automatically.
pipe = Pipeline(
    root=ROOT, phrases=PHRASES, negatives=NEGATIVES, generation=GENERATION, training=TRAINING
)
print(list(pipe.phrases))

## 4. Training data (downloads once, ~16 GB; skipped afterwards)

In [ ]:
pipe.setup_data()

### Optional: smoke test (first session only, a few minutes)
Runs every stage on tiny data in a separate folder, to catch install problems before the long runs.

In [ ]:
smoke = Pipeline(
    root=Path("/workspace/wakeword-smoke"),
    phrases=PHRASES,
    negatives=NEGATIVES,
    generation=dict(
        GENERATION,
        n_samples=20,
        n_samples_val=10,
        n_background_samples=10,
        n_background_samples_val=5,
    ),
    training=dict(TRAINING, steps=200),
    shared_data=pipe.data_dir,  # reuse the 16 GB download
)
smoke.run("B")
print("smoke test passed")

## 5. Variant A: livekit's default augmentation
The first run includes speech synthesis for both models (the longest part).

In [ ]:
pipe.run("A")

## 6. Variant B: realistic, level-safe augmentation
Reuses A's speech synthesis; only augmentation and training run.

In [ ]:
pipe.run("B")

## 7. Variant C: your room's background noise (later)
1. **Record** 20-30 min of your dorm with the microphone the detector will use: silence, fan/AC, music at a
   normal volume, a roommate talking, a video playing. Several files are fine; any of wav, m4a, mp3, flac, ogg.
2. **Upload** them to `/workspace/wakeword/noise/uploads/room/` (the folder already exists), either
   - in JupyterLab: open that folder in the file browser and drag the files in, or
   - from the Mac's terminal: `runpodctl send <folder>`, then in a pod terminal
     `cd /workspace/wakeword/noise/uploads/room && runpodctl receive <code>`.
3. Run the cell below. `prepare_noise` converts them to 16 kHz mono (livekit doesn't resample background
   audio). Adding recordings later only converts the new files, and the next `pipe.run("C")` automatically
   redoes C's augmentation and training with them (speech synthesis is still reused).

In [ ]:
pipe.prepare_noise("room")
pipe.run("C")

## 7b. Your own recordings: models M2, M3, M4
Record on the Mac with `training/record_session.py` (default `--split train`); it writes
`~/jarvie-data/real/v1/<speaker>/<wake|sleep>/` plus a `manifest.csv`. Upload the **whole `v1` folder** to
`/workspace/wakeword/real/uploads/` (drag it in, or `runpodctl send ~/jarvie-data/real/v1` and `runpodctl receive <code>`
inside `/workspace/wakeword/real/uploads/`). Never upload your `--split test` recordings: they're for evaluation only.

Each recording is trimmed to livekit's 2-second training format, copied `real_repeat` times (each copy is augmented
differently; with `speaker_aug`, copies are also speed-perturbed so your voice stands in for other voice sizes) and added
to the training clips. The other model's phrases count as negatives too. Speech synthesis is reused; only augmentation
and training run. Adding recordings later automatically reruns the runs that use them.

| | model | call |
|---|---|---|
| M1 | synthetic only (B) | `pipe.run("B")` |
| M2 | synthetic + your recordings | `pipe.run("B", real="v1")` |
| M3 | your recordings only (+ livekit's real background speech as general negatives) | `pipe.run("B", real="v1", synthetic=False)` |
| M4 | B fine-tuned on your recordings, with 10% synthetic so it doesn't forget other voices | see below |

Aim for real data at roughly 20-30% of each class: with ~300 recordings per class, `real_repeat=20` gives ~6,000 copies
next to 20,000 synthetic clips. More distinct recordings beat more copies.

In [ ]:
pipe.prepare_real("v1")  # counts per speaker and label; fails clearly if nothing was uploaded

In [ ]:
pipe.run("B", real="v1")  # M2: synthetic + your recordings
pipe.run("B", real="v1", synthetic=False)  # M3: your recordings only

In [ ]:
# M4: start from B's trained weights and fine-tune briefly at a low learning rate. Model settings
# (model_type, model_size) must match B's, or loading the weights fails with a clear error.
pipe.run(
    "B",
    trial="ft",
    real="v1",
    synthetic_share=0.1,
    init_from=("B", "base"),
    steps=5000,
    learning_rate=1e-5,
)

## 8. Hyperparameter tuning (training only)
Each trial reuses a variant's features, so it costs only training time. Use short runs to compare, then a
full-length run of the best. Training settings livekit accepts include `steps`, `learning_rate`,
`weight_decay`, `label_smoothing`, `max_negative_weight`, `target_fp_per_hour`, `model` (`model_type`:
`conv_attention`/`dnn`/`rnn`, `model_size`: `tiny`/`small`/`medium`/`large`) and `batch_n_per_class`.

In [ ]:
# Examples; give every trial a unique name. `models=["jarvie_wake"]` trains just one model.
pipe.run("B", trial="short-base", steps=20000)
pipe.run("B", trial="short-lr3e-4", steps=20000, learning_rate=3e-4)
pipe.run("B", trial="short-negw5000", steps=20000, max_negative_weight=5000)
pipe.run("B", trial="short-large", steps=20000, model={"model_size": "large"})

## 9. Results and models
`run_eval`'s numbers use synthetic speech; the real test is your own recordings in `wakeword_eval.ipynb` on the
Mac. Models are in `/workspace/wakeword/models/` as `<model>__<variant>__<trial>.onnx`. Download them by
right-clicking in JupyterLab, or `runpodctl send /workspace/wakeword/models` and `runpodctl receive <code>` on the
Mac. On the Mac, copy the one to test into `server/models/` as `jarvie_wake.onnx` / `jarvie_sleep.onnx`.

In [ ]:
cols = ["time", "model", "variant", "trial", "aut", "fpph", "recall", "optimal_threshold", "onnx"]
df = pipe.results()
df[[c for c in cols if c in df.columns]]

## 10. Housekeeping
Sizes of the cached stages (hard-linked files are counted in each folder that links them).

In [ ]:
pipe.disk_usage()